In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp,col

spark = SparkSession.builder.getOrCreate()

tables = [ "customers","sales","sales_orders"]

source_path=  "/Volumes/ecommerce_analytics/bronze/raw_data/"

catalog = "ecommerce_analytics"

schema= "bronze"

# Utility functions
 

def add_metadata_columns(df):
     df=df.withColumn ("last_updates_ts", current_timestamp())\
        .withColumn("file_path", col("_metadata.file_path"))
     return df

for table in tables:
    print(f"processing table{table}")
    input_path= f"{source_path}{table}/"
    df =spark.read.csv(input_path, inferSchema=True , header=True)
    print(f"read completed for{table}, Nowadding metadata comlumns")
    df = add_metadata_columns(df)
    df.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(f"{catalog}.{schema}.{table}")
    print(f"write complted for {table}")
    print(f"write completed fro all {table}")